# **Bayes' Theorem: What Does a Rain Alert Really Tell Us?**

**Setup:** Python 3; no external packages required.

You are about to walk to campus. Your weather app sends a rain alert. You need to decide whether to bring an umbrella. The app's alert rate alone does not answer that question: it tells us how often the app alerts when it rains, not how often it rains after an alert. We will use Bayes' theorem to answer the question that matters for your decision: **given this alert, how likely is rain?**

To learn the calculation, we will imagine the campus has a record of past mornings. The numbers are deliberately supplied as facts in this exercise, not discovered by Python and not a real forecast. Counting a convenient 1,000 mornings makes the percentages easier to see; the same answer would result from the same proportions in a smaller or larger record.

**Learning goals:** answer the umbrella question from evidence, explain every probability in the calculation, and choose an action by comparing its expected costs.

## 1. Why count past mornings?

We want to make a decision **today**, but we do not yet know whether it will rain. A record of similar past mornings gives us two useful things: how common rain usually is, and how often the app's alert has been right or wrong. For this exercise, suppose that record contains 1,000 mornings and says:

- It rained on 300 mornings and stayed dry on 700. Thus the historical rain rate is $300/1000=0.30$, or 30%. This is where `rain_rate` comes from: it is a **given fact about the imagined past record**, not an unexplained constant and not a claim about today's weather.
- On the 300 rainy mornings, the app sent an alert on 80% of them: $0.80\times300=240$ alerts.
- On the 700 dry mornings, the app still sent an alert on 25% of them: $0.25\times700=175$ false alarms.

There were $240+175=415$ alerts altogether. Among those alert mornings, 240 were rainy. So, when the app alerts, the fraction of alert mornings that were rainy is $240/415\approx57.8\%$.

That is useful for the campus walk: it estimates the chance you face rain **after seeing the alert**, which you can compare with the inconvenience of carrying an umbrella. The alert's 80% rate is not the answer; it describes the different group of mornings when it was already raining. The lesson is about using the past record to answer today's question.

### The record at a glance

The same 1,000 mornings, split by weather and by whether the app sent an alert:

<table frame="hsides" rules="rows" style="border-collapse:collapse;border-left:none;border-right:none">
<tr><th style="text-align:left">Morning type</th><th style="text-align:right">Mornings</th><th style="text-align:right">Alert sent</th><th style="text-align:right">No alert</th></tr>
<tr><td style="text-align:left">Rainy</td><td style="text-align:right">300</td><td style="text-align:right">240</td><td style="text-align:right">60</td></tr>
<tr><td style="text-align:left">Dry</td><td style="text-align:right">700</td><td style="text-align:right">175</td><td style="text-align:right">525</td></tr>
<tr><td style="text-align:left"><b>Total</b></td><td style="text-align:right"><b>1,000</b></td><td style="text-align:right"><b>415</b></td><td style="text-align:right"><b>585</b></td></tr>
</table>

Look only at the **Alert sent** column: 240 of its 415 mornings were rainy.

In [ ]:
# Given facts in the hypothetical historical record described above.
historical_mornings = 1_000
historical_rainy_mornings = 300
alert_rate_on_rainy_mornings = 0.80
alert_rate_on_dry_mornings = 0.25

# The 30% prior is calculated from the historical counts: 300 rainy / 1,000 total.
rain_rate = historical_rainy_mornings / historical_mornings
rainy_mornings = historical_mornings * rain_rate
dry_mornings = historical_mornings - rainy_mornings
alerts_on_rainy_mornings = rainy_mornings * alert_rate_on_rainy_mornings
alerts_on_dry_mornings = dry_mornings * alert_rate_on_dry_mornings
all_alerts = alerts_on_rainy_mornings + alerts_on_dry_mornings
rain_given_alert = alerts_on_rainy_mornings / all_alerts

print(f"Historical rain rate (prior): {rain_rate:.0%}")
print(f"Rainy mornings in the record: {rainy_mornings:.0f}")
print(f"Alerts on rainy mornings: {alerts_on_rainy_mornings:.0f}")
print(f"Alerts on dry mornings (false alarms): {alerts_on_dry_mornings:.0f}")
print(f"Total alerts: {all_alerts:.0f}")
print(f"Chance of rain on an alert morning: {rain_given_alert:.1%}")

### The first surprising result

The app catches most rainy mornings, but it also sends alerts on dry mornings. In our 1,000 mornings, 240 alerts were useful and 175 were false alarms. So the chance of rain **after** receiving an alert is $240/415\approx57.8\%$—not 80%.

Why isn't it 80%? Because 80% answers a different question: “If it is already a rainy morning, how often does the app alert?” We ask the opposite practical question: “If the app alerts, how often is it rainy?” The groups being counted have different starting points.

To make the notation readable, let $R$ mean **rain** and $A$ mean **app sends an alert**. The vertical bar $\mid$ means “looking only at cases where” or “given.” Thus $P(R\mid A)$ means “the fraction of alert mornings that are rainy.” By contrast, $P(A\mid R)$ means “the fraction of rainy mornings that get an alert.” The bar does not mean division; the fraction is how we calculate that conditional probability.

Before seeing an alert, the **prior** chance of rain is $P(R)=300/1000=0.30$. The app's alert rate on rainy mornings is $P(A\mid R)=0.80$; its alert rate on dry mornings is $P(A\mid\text{no rain})=0.25$. These probabilities refer to three different groups: all mornings, rainy mornings, and dry mornings.

## 2. Now translate the counting into Bayes' theorem

A probability such as $P(R\cap A)$ is the chance that **both** rain and an alert occur on the same morning. The symbol $\cap$ means “and at the same time.” We use it instead of a comma so the meaning is visible.

$$
P(R\mid A)=\frac{P(R\cap A)}{P(A)},\qquad P(A)>0.
$$

Read this as: **among all alert mornings**, count those that were rainy. The condition $P(A)>0$ says that there must be at least some alert mornings; if there were zero alerts, “the fraction of alert mornings that are rainy” would have no denominator and would be undefined.

Bayes' theorem is not a mysterious reversal. It follows by counting the **same both-rain-and-alert mornings** in two ways:

1. Start with all mornings and ask what fraction are both rainy and alert mornings:

$$P(R\cap A)=P(A\mid R)P(R).$$

This says: the chance it rains **and** there is an alert equals the chance it rains, multiplied by the fraction of rainy mornings that get an alert.

2. Start instead with alert mornings and ask what fraction of all mornings are both rainy and alert mornings:

$$P(R\cap A)=P(R\mid A)P(A).$$

Both expressions equal the same quantity, so set them equal and divide by $P(A)$:

$$
P(R\mid A)=\frac{P(A\mid R)P(R)}{P(A)}.
$$

This is Bayes' theorem. The phrase “reverse the condition” is easy to misunderstand: the theorem does not claim $P(R\mid A)=P(A\mid R)$. It gives a way to calculate the first from the second, while accounting for how common rain and false alerts are.

We still need the denominator: how common are alerts overall? Alerts come from exactly two groups—rainy mornings and dry mornings—so add the alerts contributed by each group:

$$
P(A)=P(A\mid R)P(R)+P(A\mid \text{no rain})P(\text{no rain}).
$$

Here “no rain” is written in words—there is no superscript to memorize. Substituting this into Bayes' theorem gives the full calculation:

$$
P(R\mid A)=\frac{P(A\mid R)P(R)}{P(A\mid R)P(R)+P(A\mid\text{no rain})P(\text{no rain})}.
$$

The denominator is the total number (or probability) of **all alerts**: correct alerts plus false alarms. That is why the base rate $P(R)$ matters.

In [ ]:
prior_rain = 0.30
alert_rate_on_rain = 0.80
alert_rate_on_dry = 0.25

chance_of_any_alert = (
    alert_rate_on_rain * prior_rain
    + alert_rate_on_dry * (1 - prior_rain)
)
chance_of_rain_given_alert = (
    alert_rate_on_rain * prior_rain / chance_of_any_alert
)

print(f"P(alert) = {chance_of_any_alert:.3f}")
print(f"P(rain | alert) = {chance_of_rain_given_alert:.1%}")
assert abs(chance_of_rain_given_alert - 240 / 415) < 1e-12

In [ ]:
prior_rain = 0.30
alert_rate_if_rain = 0.80
alert_rate_if_dry = 0.25

rain_and_alert = prior_rain * alert_rate_if_rain
dry_and_alert = (1 - prior_rain) * alert_rate_if_dry
all_alerts = rain_and_alert + dry_and_alert
posterior_rain = rain_and_alert / all_alerts

print(f"Rain AND alert: {rain_and_alert:.3f}")
print(f"No rain AND alert: {dry_and_alert:.3f}")
print(f"All alerts: {all_alerts:.3f}")
print(f"Rain given alert: {posterior_rain:.1%}")
assert abs(posterior_rain - 0.24 / 0.415) < 1e-12

## 3. From the updated probability to a decision

A probability tells us how plausible rain is; it does not by itself decide whether to carry an umbrella. We must also consider what each mistake costs. Suppose carrying an umbrella is mildly inconvenient, with a cost of 2 points if it stays dry. Getting soaked without an umbrella costs 4 points. For this small example, assume the umbrella prevents getting soaked.

Let $p$ mean “the chance of rain **given that the app sent an alert**.” Every alert morning is either rainy or dry, so the chance of a dry morning is $1-p$. The average (expected) cost is a probability-weighted average: bringing the umbrella costs 2 points on dry mornings and 0 on rainy ones, so its average cost is $2(1-p)$. Leaving it at home costs 4 points on rainy mornings and 0 on dry ones, so its average cost is $4p.

$$
\text{Average cost of bringing} = 2(1-p),\qquad
\text{Average cost of leaving it} = 4p.
$$

Choose the umbrella when its average cost is smaller. Solve the inequality one step at a time:

$$
\begin{aligned}
2(1-p)&<4p \\
2&<6p \\
p&>\frac{1}{3}.
\end{aligned}
$$

So bring the umbrella if the chance of rain is greater than one third (about 33.3%). After an app alert, our count gave $p=240/415\approx57.8\%$, which is above that threshold. The app alert therefore makes bringing the umbrella the lower-cost choice in this toy example. The threshold comes from the costs; Bayes' theorem supplied the updated probability.

## Individual practice: what if rain is more or less common?

Keep the app's behavior fixed: it alerts on 80% of rainy mornings and 25% of dry mornings. Change only how common rainy mornings are. This separates the **prior** (how common rain is before the alert) from the **evidence** (the alert).

### Tasks

1. Before running the code, predict whether an alert is more or less convincing when rain is rare.
2. Run the code for prior probabilities of 10%, 30%, and 60%. Record the chance of rain after an alert in each case.
3. For each prior, decide whether to bring the umbrella using the one-third cost threshold. Explain any case where the decision differs.
4. In your own words, explain why the app's 80% alert rate on rainy mornings is not the same as the chance of rain after receiving an alert.

In [ ]:
alert_rate_on_rainy_mornings = 0.80
alert_rate_on_dry_mornings = 0.25

def rain_chance_after_alert(prior_rain):
    """Calculate the chance of rain after an alert, using Bayes' theorem."""
    chance_of_any_alert = (
        alert_rate_on_rainy_mornings * prior_rain
        + alert_rate_on_dry_mornings * (1 - prior_rain)
    )
    return alert_rate_on_rainy_mornings * prior_rain / chance_of_any_alert

def should_bring_umbrella(chance_of_rain):
    """Return the lower-cost choice for inconvenience=2 and getting-soaked=4."""
    cost_of_bringing = 2 * (1 - chance_of_rain)
    cost_of_leaving = 4 * chance_of_rain
    return cost_of_bringing < cost_of_leaving

for prior in (0.10, 0.30, 0.60):
    posterior = rain_chance_after_alert(prior)
    print(
        f"Prior rain chance: {prior:.0%}; "
        f"rain chance after alert: {posterior:.1%}; "
        f"bring umbrella: {should_bring_umbrella(posterior)}"
    )

In [ ]:
assert abs(rain_chance_after_alert(0.10) - (0.08 / 0.305)) < 1e-12
assert abs(rain_chance_after_alert(0.30) - (0.24 / 0.415)) < 1e-12
assert abs(rain_chance_after_alert(0.60) - (0.48 / 0.58)) < 1e-12
assert should_bring_umbrella(rain_chance_after_alert(0.10)) is False
assert should_bring_umbrella(rain_chance_after_alert(0.30)) is True
assert should_bring_umbrella(rain_chance_after_alert(0.60)) is True
print("Posterior calculations and umbrella decisions passed.")

### Check for understanding

Without quoting the formula, explain what the numerator counts, what the denominator counts, why false alarms appear in that denominator, and why the prior probability changes the answer.